# 07 One-Click LexAI Runner (Hardened)

This notebook is designed to run end-to-end with robust fallbacks for Databricks workspace notebook paths.

Run order: `Cell 1 -> Cell 10`


In [0]:
# CELL 1: Runtime Flags
AUTO_INSTALL_MISSING = True
RUN_SMOKE_TEST = True
START_FASTAPI = True
START_STREAMLIT = False

FASTAPI_PORT = 8000
STREAMLIT_PORT = 8501

# Set explicitly if auto-detection fails:
REPO_DIR_OVERRIDE = "/Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform"

SMOKE_TEST_QUERIES = [
    "Penalty for not wearing helmet in short within 120 words",
    "What does section 129 say in detail within 180 words",
]

print("[CELL 1] Flags loaded")
print({
    "AUTO_INSTALL_MISSING": AUTO_INSTALL_MISSING,
    "RUN_SMOKE_TEST": RUN_SMOKE_TEST,
    "START_FASTAPI": START_FASTAPI,
    "START_STREAMLIT": START_STREAMLIT,
    "FASTAPI_PORT": FASTAPI_PORT,
    "STREAMLIT_PORT": STREAMLIT_PORT,
    "REPO_DIR_OVERRIDE": REPO_DIR_OVERRIDE,
})


[CELL 1] Flags loaded
{'AUTO_INSTALL_MISSING': True, 'RUN_SMOKE_TEST': True, 'START_FASTAPI': True, 'START_STREAMLIT': False, 'FASTAPI_PORT': 8000, 'STREAMLIT_PORT': 8501, 'REPO_DIR_OVERRIDE': '/Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform'}


In [0]:
# CELL 2: Resolve repo path safely
import os
import sys
from pathlib import Path
from datetime import datetime


def log(msg: str):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")


def _repo_has_required_files(repo_dir: Path) -> bool:
    return (repo_dir / "apps" / "fastapi_app.py").exists() and (repo_dir / "apps" / "lexai06_notebook_adapter.py").exists()


def _safe_walk_for_repo(root: Path):
    skip_dirs = {"__pycache__", ".git", ".ipynb_checkpoints"}

    def _onerror(_err):
        return None

    for dirpath, dirnames, _filenames in os.walk(root, topdown=True, onerror=_onerror):
        dirnames[:] = [d for d in dirnames if d not in skip_dirs]
        p = Path(dirpath)
        try:
            if _repo_has_required_files(p):
                return p
        except Exception:
            continue
    return None


def _context_repo_guess():
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        nb_path = ctx.notebookPath().get()  # /Users/<email>/<repo>/notebooks/07_...
        if not nb_path:
            return None
        pp = Path(nb_path)
        # repo likely parent.parent
        ws_repo = Path("/Workspace") / Path(*pp.parent.parent.parts[1:])
        if ws_repo.exists() and _repo_has_required_files(ws_repo):
            return ws_repo
    except Exception:
        pass
    return None


def resolve_repo_dir() -> Path:
    if REPO_DIR_OVERRIDE and str(REPO_DIR_OVERRIDE).strip():
        p = Path(REPO_DIR_OVERRIDE.strip())
        if p.exists() and _repo_has_required_files(p):
            return p

    cwd = Path(os.getcwd()).resolve()
    for cand in [cwd] + list(cwd.parents):
        try:
            if _repo_has_required_files(cand):
                return cand
        except Exception:
            continue

    g = _context_repo_guess()
    if g is not None:
        return g

    for root in [Path("/Workspace/Repos"), Path("/Workspace/Users"), Path("/Workspace")]:
        if not root.exists():
            continue
        hit = _safe_walk_for_repo(root)
        if hit is not None:
            return hit

    raise FileNotFoundError(
        "Could not locate repo root. Set REPO_DIR_OVERRIDE to your repo path."
    )


REPO_DIR = resolve_repo_dir()
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

log(f"Repo root: {REPO_DIR}")
print("[CELL 2] OK")


[16:53:35] Repo root: /Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform
[CELL 2] OK


In [0]:
# CELL 3: Dependency preflight (no Python restart)
import importlib
import importlib.metadata as ilm
import subprocess

REQ_FILE = Path("apps/requirements.txt")
if not REQ_FILE.exists():
    raise FileNotFoundError(f"Missing requirements file: {REQ_FILE}")

required_packages = [
    "fastapi",
    "uvicorn",
    "streamlit",
    "requests",
    "pydantic",
    "sentence-transformers",
    "transformers",
    "accelerate",
    "mlflow",
    "databricks-sdk",
    "typing_extensions>=4.6.0",
]


def _pkg_name(spec: str) -> str:
    for sep in [">=", "==", "<=", "~=", ">", "<"]:
        if sep in spec:
            return spec.split(sep)[0].strip()
    return spec.strip()


missing_specs = []
for spec in required_packages:
    name = _pkg_name(spec)
    try:
        ilm.version(name)
    except Exception:
        missing_specs.append(spec)

print("[CELL 3] Missing specs:", missing_specs)

if missing_specs and AUTO_INSTALL_MISSING:
    cmd = [sys.executable, "-m", "pip", "install", "-q", "-r", str(REQ_FILE)] + missing_specs
    print("[CELL 3] Installing missing packages...")
    subprocess.check_call(cmd)
    print("[CELL 3] Installation complete")
elif missing_specs and not AUTO_INSTALL_MISSING:
    raise RuntimeError(f"Missing packages: {missing_specs}. Set AUTO_INSTALL_MISSING=True")

# Hard validation for TypeIs support without forcing restart.
try:
    import typing_extensions
    importlib.reload(typing_extensions)
    _ = typing_extensions.TypeIs
    print("[CELL 3] typing_extensions.TypeIs available")
except Exception as e:
    raise RuntimeError(
        "typing_extensions TypeIs still unavailable. Run: dbutils.library.restartPython(), then rerun from Cell 1"
    ) from e

print("[CELL 3] OK")


[CELL 3] Missing specs: ['fastapi', 'uvicorn', 'streamlit', 'sentence-transformers', 'transformers', 'accelerate', 'mlflow']
[CELL 3] Installing missing packages...


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyter-server 1.23.4 requires anyio<4,>=3.1.0, but you have anyio 4.12.1 which is incompatible.
googleapis-common-protos 1.62.0 requires protobuf!=3.20.0,!=3.20.1,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0.dev0,>=3.19.5, but you have protobuf 6.33.5 which is incompatible.

[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


[CELL 3] Installation complete
[CELL 3] typing_extensions.TypeIs available
[CELL 3] OK


In [0]:
# CELL 4: Spark / cluster context
from pyspark.sql import SparkSession

spark = SparkSession.getActiveSession() or SparkSession.builder.getOrCreate()
print("[CELL 4] Spark session ready:", bool(spark))

cluster_id = "unknown"
org_id = "unknown"
workspace_url = "unknown"

for key, var in [
    ("spark.databricks.clusterUsageTags.clusterId", "cluster_id"),
    ("spark.databricks.clusterUsageTags.orgId", "org_id"),
    ("spark.databricks.workspaceUrl", "workspace_url"),
]:
    try:
        val = spark.conf.get(key)
        if var == "cluster_id":
            cluster_id = val
        elif var == "org_id":
            org_id = val
        elif var == "workspace_url":
            workspace_url = val
    except Exception:
        pass

print("[CELL 4] cluster_id:", cluster_id)
print("[CELL 4] org_id:", org_id)
print("[CELL 4] workspace_url:", workspace_url)


[CELL 4] Spark session ready: True


2026-03-01 16:58:02,167 26025 ERROR _handle_rpc_error GRPC Error received
Traceback (most recent call last):
  File "/databricks/python/lib/python3.10/site-packages/pyspark/sql/connect/client/core.py", line 1724, in config
    resp = self._stub.Config(req, metadata=self.metadata())
  File "/databricks/python/lib/python3.10/site-packages/grpc/_interceptor.py", line 277, in __call__
    response, ignored_call = self._with_call(
  File "/databricks/python/lib/python3.10/site-packages/grpc/_interceptor.py", line 332, in _with_call
    return call.result(), call
  File "/databricks/python/lib/python3.10/site-packages/grpc/_channel.py", line 439, in result
    raise self
  File "/databricks/python/lib/python3.10/site-packages/grpc/_interceptor.py", line 315, in continuation
    response, call = self._thunk(new_method).with_call(
  File "/databricks/python/lib/python3.10/site-packages/grpc/_channel.py", line 1193, in with_call
    return _end_unary_response_blocking(state, call, True, None)
 

[CELL 4] cluster_id: 0301-162856-vrue684y-v2n
[CELL 4] org_id: unknown
[CELL 4] workspace_url: dbc-afb2e98d-d930.cloud.databricks.com


In [0]:
# CELL 4.5: Fix circular import in accelerate
import sys

# Clear any partially initialized accelerate/transformers modules
modules_to_clear = [k for k in sys.modules.keys() if k.startswith(('accelerate', 'transformers', 'torch'))]
for mod in modules_to_clear:
    del sys.modules[mod]

print(f"[CELL 4.5] Cleared {len(modules_to_clear)} cached modules")

# Reinstall compatible versions
import subprocess
cmd = [
    sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "accelerate>=0.20.0",
    "transformers>=4.30.0"
]
print("[CELL 4.5] Reinstalling accelerate and transformers...")
subprocess.check_call(cmd)

print("[CELL 4.5] Restarting Python to clear import state...")
dbutils.library.restartPython()

[CELL 4.5] Cleared 1117 cached modules
[CELL 4.5] Reinstalling accelerate and transformers...



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


[CELL 4.5] Restarting Python to clear import state...


In [0]:
# CELL 5: Initialize notebook-06 engine through adapter
from pathlib import Path
import importlib
import os
import apps.lexai06_notebook_adapter as _adapter

importlib.reload(_adapter)
NotebookEngine = _adapter.NotebookEngine

workspace_candidates = [
    "/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/notebooks/06_High-precision_QA_Legal_Reasoning_Engine",
    "/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/apps/notebook_06_snapshot",
    "/Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/notebooks/06_High-precision_QA_Legal_Reasoning_Engine",
    "/Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/apps/notebook_06_snapshot",
    str(Path(REPO_DIR) / "notebooks" / "06_High-precision_QA_Legal_Reasoning_Engine.ipynb"),
    str(Path(REPO_DIR) / "apps" / "notebook_06_snapshot.ipynb"),
]

print("[CELL 5] Notebook candidates:")
for c in workspace_candidates:
    try:
        print(" -", c, "exists=", Path(c).exists())
    except Exception:
        print(" -", c, "exists=ERROR")

status = None
last_err = None
for cand in workspace_candidates:
    try:
        os.environ["LEXAI06_NOTEBOOK_PATH"] = cand
        engine = NotebookEngine(notebook_path=Path(cand))
        status = engine.initialize()
        print(f"[CELL 5] Initialized using candidate: {cand}")
        break
    except Exception as e:
        last_err = e
        print(f"[CELL 5] Candidate failed: {cand} -> {e}")

if status is None:
    raise RuntimeError(f"Engine initialization failed for all candidates. Last error: {last_err}")

print("[CELL 5] Engine initialized")
for k, v in status.items():
    print(f"  - {k}: {v}")

if not status.get("ready"):
    raise RuntimeError(f"Engine failed to initialize: {status}")


[CELL 5] Notebook candidates:
 - /Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/notebooks/06_High-precision_QA_Legal_Reasoning_Engine exists= False
 - /Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/apps/notebook_06_snapshot exists= False
 - /Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/notebooks/06_High-precision_QA_Legal_Reasoning_Engine exists= True
 - /Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/apps/notebook_06_snapshot exists= True
 - /Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/notebooks/06_High-precision_QA_Legal_Reasoning_Engine.ipynb exists= False
 - /Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/apps/notebook_06_snapshot.ipynb exists= False


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

[16:55:04] Using endpoint backend: databricks-meta-llama-3-3-70b-instruct
[16:55:13] Loaded lexical artifacts from Delta for signature=312b53f785d939bab1c3.
--- Runtime Status ---
Data signature: 312b53f785d939bab1c3
Embedding source: /Volumes/workspace/legal_data/vector_db_test/legal_embeddings_delta
Records loaded: 5194
Embedding dim: 384
Avg doc len: 183.25
Vocabulary size: 13196
Lexical source: delta_artifact
Embedder: sentence-transformers/all-MiniLM-L6-v2
Reranker: cross-encoder/ms-marco-MiniLM-L-6-v2
LLM backend: endpoint (databricks-meta-llama-3-3-70b-instruct)
[CELL 5] Initialized using candidate: /Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/notebooks/06_High-precision_QA_Legal_Reasoning_Engine
[CELL 5] Engine initialized
  - ready: True
  - error: 
  - notebook_path: /Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/notebooks/06_High-precision_QA_Legal_Reasoning_Engine
  - resolved_notebook_path: /tmp/lexai06_exported_from_workspace.ipynb
  - exec_cell_nu

In [0]:
# CELL 6: Smoke test (optional)
if RUN_SMOKE_TEST:
    print("[CELL 6] Running smoke tests...")
    for idx, q in enumerate(SMOKE_TEST_QUERIES, start=1):
        print("=" * 90)
        print(f"[{idx}] Query: {q}")
        out = engine.answer_query(q)
        print("Mode:", out.get("mode"))
        print("Source:", out.get("source"))
        print("Confidence:", out.get("confidence"))
        print("Sections:", out.get("sections", []))
        print("Citations:", out.get("citations", [])[:5])
        print("Latency:", out.get("latency_ms", {}))
        print("Answer:")
        print(out.get("answer", ""))
    print("[CELL 6] Smoke tests done")
else:
    print("[CELL 6] RUN_SMOKE_TEST=False -> skipped")


[CELL 6] Running smoke tests...
[1] Query: Penalty for not wearing helmet in short within 120 words
Mode: rule_based
Source: traffic_rules
Confidence: top_score=0.617, avg_lex=0.732, coverage=0.444, normative_hits=5, judgment_hits=0, candidates=2512, shortlist=140, route=statute_strict
Sections: ['177', '178', '179', '180', '181', '182', '183', '184', '186', '189', '190', '191', '129']
Citations: ['Motor Vehicles Act 1988 - Section Chapter V', 'Motor Vehicles Act 1988 - Section Chapter VII', 'Motor Vehicle Ammendment Act 2019 - Section Chapter XI', 'Motor Vehicles Act 1988 - Section Chapter VI', 'Central Motor Vehicle Rules 1989 - Section Chapter VI']
Latency: {'candidate_fetch_ms': 3.48, 'routing_ms': 62.06, 'lexical_ms': 137.36, 'embed_ms': 64.51, 'dense_ms': 7.75, 'rrf_ms': 0.28, 'rerank_ms': 1054.8, 'retrieve_total_ms': 1333.4, 'generation_ms': 0.0, 'total_ms': 1336.26}
Answer:
Law:
Section 129 of the Motor Vehicles Act requires riders to wear protective headgear while riding two-w

In [0]:
# CELL 7: Start FastAPI server (background thread)
import threading
import uvicorn

FASTAPI_SERVER = globals().get("FASTAPI_SERVER")
FASTAPI_THREAD = globals().get("FASTAPI_THREAD")

if START_FASTAPI:
    if FASTAPI_THREAD is not None and FASTAPI_THREAD.is_alive():
        print(f"[CELL 7] FastAPI already running on port {FASTAPI_PORT}")
    else:
        from apps.fastapi_app import app
        config = uvicorn.Config(app, host="0.0.0.0", port=int(FASTAPI_PORT), log_level="info")
        FASTAPI_SERVER = uvicorn.Server(config)
        FASTAPI_THREAD = threading.Thread(target=FASTAPI_SERVER.run, daemon=True)
        FASTAPI_THREAD.start()
        globals()["FASTAPI_SERVER"] = FASTAPI_SERVER
        globals()["FASTAPI_THREAD"] = FASTAPI_THREAD
        print(f"[CELL 7] FastAPI started on 0.0.0.0:{FASTAPI_PORT}")

    print("[CELL 7] Local health URL:", f"http://127.0.0.1:{FASTAPI_PORT}/health")

    if workspace_url != "unknown" and org_id != "unknown" and cluster_id != "unknown":
        proxy_url = f"https://{workspace_url}/driver-proxy/o/{org_id}/{cluster_id}/{FASTAPI_PORT}/health"
        print("[CELL 7] Driver proxy URL:", proxy_url)
else:
    print("[CELL 7] START_FASTAPI=False -> skipped")


[CELL 7] FastAPI started on 0.0.0.0:8000
[CELL 7] Local health URL: http://127.0.0.1:8000/health


INFO:     Started server process [26025]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [0]:
# CELL 8: FastAPI smoke call (optional)
import requests

if START_FASTAPI:
    try:
        h = requests.get(f"http://127.0.0.1:{FASTAPI_PORT}/health", timeout=30)
        print("[CELL 8] /health status:", h.status_code)
        print(h.json())

        payload = {
            "query": "What is the penalty for not wearing a helmet?",
            "style": "short",
            "word_limit": 120,
        }
        r = requests.post(f"http://127.0.0.1:{FASTAPI_PORT}/v1/legal/answer", json=payload, timeout=180)
        print("[CELL 8] /v1/legal/answer status:", r.status_code)
        try:
            body = r.json()
            print("[CELL 8] answer preview:", body.get("answer", "")[:500])
        except Exception:
            print("[CELL 8] raw response:", r.text[:500])
    except Exception as e:
        print("[CELL 8] API call failed:", e)
else:
    print("[CELL 8] START_FASTAPI=False -> skipped")


[CELL 8] API call failed: HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /health (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7fbf7761d750>: Failed to establish a new connection: [Errno 111] Connection refused'))


In [0]:
# CELL 9: Optional Streamlit start (blocking)
import subprocess

if START_STREAMLIT:
    os.environ["LEXAI_API_BASE_URL"] = f"http://127.0.0.1:{FASTAPI_PORT}"
    cmd = [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        "apps/streamlit_app.py",
        "--server.port", str(STREAMLIT_PORT),
        "--server.address", "0.0.0.0",
    ]
    print("[CELL 9] Starting Streamlit:", " ".join(cmd))
    if workspace_url != "unknown" and org_id != "unknown" and cluster_id != "unknown":
        ui_url = f"https://{workspace_url}/driver-proxy/o/{org_id}/{cluster_id}/{STREAMLIT_PORT}/"
        print("[CELL 9] Streamlit URL:", ui_url)
    subprocess.call(cmd)
else:
    print("[CELL 9] START_STREAMLIT=False -> skipped")


[CELL 9] START_STREAMLIT=False -> skipped


In [0]:
# CELL 10: Stop helper
if "FASTAPI_SERVER" in globals() and globals().get("FASTAPI_SERVER") is not None:
    globals()["FASTAPI_SERVER"].should_exit = True
    print("[CELL 10] FastAPI stop requested")
else:
    print("[CELL 10] FastAPI was not running")


[CELL 10] FastAPI stop requested


INFO:     Shutting down
